In [3]:
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

# 1. Apple Silicon (MPS) & CUDA Support
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"🚀 Training CNN on Device: {device}")

# Paths
JSONL_PATH = "/Users/saimohith/Documents/Sem-5/DL/BoneFractureQA/metadata.jsonl"
MODEL_SAVE_PATH = "/Users/saimohith/Documents/Sem-5/DL/BoneFractureQA/efficientnet_fracture.pth"

cnn_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CNNFractureDataset(Dataset):
    def __init__(self, jsonl_path, split='train', transform=None):
        self.dataset = []
        self.transform = transform
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                if data['split'] == split:
                    self.dataset.append(data)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        label = 1.0 if "Yes" in item['answer'] else 0.0
        try:
            image = Image.open(item['image']).convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224), color='white')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor([label], dtype=torch.float32)

if __name__ == "__main__":
    print("📦 Loading Datasets...")
    train_dataset = CNNFractureDataset(JSONL_PATH, split='train', transform=cnn_transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    print("🏗️ Building EfficientNet-B0...")
    class EfficientNetClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
            for param in self.model.parameters():
                param.requires_grad = False
            in_features = self.model.classifier[1].in_features
            self.model.classifier = nn.Sequential(
                nn.Dropout(p=0.3, inplace=True),
                nn.Linear(in_features, 1)
            )
        def forward(self, x):
            return self.model(x)

    model_cnn = EfficientNetClassifier().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model_cnn.model.classifier.parameters(), lr=1e-3)

    epochs = 3
    print(f"🔥 Starting CNN Training for {epochs} Epochs...")
    for epoch in range(epochs):
        model_cnn.train()
        total_loss = 0
        for idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model_cnn(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            if idx % 10 == 0:
                print(f"   Epoch: {epoch+1}/{epochs} | Batch: {idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    torch.save(model_cnn.state_dict(), MODEL_SAVE_PATH)
    print(f"✅ CNN Backbone Saved to {MODEL_SAVE_PATH}")

🚀 Training CNN on Device: mps
📦 Loading Datasets...
🏗️ Building EfficientNet-B0...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/saimohith/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)>